# Cálculo da Suscetibilidade, probabilidade e perigosidade

Integração das variáveis preparadas anteriormente para calcular a suscetibilidade, a probabilidade e a perigosidade de incêndio na área de estudo.

In [ ]:
from glass.ete.lri import HeuristicLri
import os

import sys

sys.path.append("/code/scripts")

from lri_sum import HeuristicLriCount

In [ ]:
# ALTERAR APENAS ESTAS DUAS VARIÁVEIS

scenario = "C5"
area = "extremadura"


base = f"/code/data/processed/{area}"
ws = f"/code/data/results/{area}/{scenario}/final"


# Variáveis topográficas
_vars = {
    "dem": (
        f"{base}/topo/reclassified/"
        f"rcls_dem_{area}.tif"
    ),
    "slope": (
        f"{base}/topo/reclassified/"
        f"rcls_slope_{area}.tif"
    ),
}


# C1 — Centro | ICNF | 1995–2024
if scenario == "C1":
    train_start = 1995
    train_end = 2024

    ba_dir = f"{base}/area_ardida/icnf"

    bareas = (
        f"{ba_dir}/raster_count/"
        f"rst_ba_1995_2024.tif"
    )

    bareas_by_year = f"{ba_dir}/train"

    lulcs = [
        f"{base}/lulc/rasters/lulc_1995.tif",
        f"{base}/lulc/rasters/lulc_2007.tif",
        f"{base}/lulc/rasters/lulc_2010.tif",
        f"{base}/lulc/rasters/lulc_2015.tif",
        f"{base}/lulc/rasters/lulc_2018.tif",
    ]

    burncos = [
        f"{ba_dir}/raster_count/rst_ba_1995_2006.tif",
        f"{ba_dir}/raster_count/rst_ba_2007_2009.tif",
        f"{ba_dir}/raster_count/rst_ba_2010_2014.tif",
        f"{ba_dir}/raster_count/rst_ba_2015_2017.tif",
        f"{ba_dir}/raster_count/rst_ba_2018_2024.tif",
    ]

    weights = [12, 3, 5, 3, 7]


# C3 — Centro | ICNF | 2008–2024
elif scenario == "C3":
    train_start = 2008
    train_end = 2024

    ba_dir = f"{base}/area_ardida/icnf"

    bareas = (
        f"{ba_dir}/raster_count/"
        f"rst_ba_2008_2024.tif"
    )

    bareas_by_year = f"{ba_dir}/train"

    lulcs = [
        f"{base}/lulc/rasters/lulc_2007.tif",
        f"{base}/lulc/rasters/lulc_2010.tif",
        f"{base}/lulc/rasters/lulc_2015.tif",
        f"{base}/lulc/rasters/lulc_2018.tif",
    ]

    burncos = [
        f"{ba_dir}/raster_count/rst_ba_2008_2009.tif",
        f"{ba_dir}/raster_count/rst_ba_2010_2014.tif",
        f"{ba_dir}/raster_count/rst_ba_2015_2017.tif",
        f"{ba_dir}/raster_count/rst_ba_2018_2024.tif",
    ]

    weights = [2, 5, 3, 7]


# C5 — Centro ou Extremadura | EFFIS | 2008–2024
elif scenario == "C5":
    train_start = 2008
    train_end = 2024

    ba_dir = f"{base}/area_ardida/effis"

    bareas = (
        f"{ba_dir}/raster_count/"
        f"rst_ba_2008_2024.tif"
    )

    bareas_by_year = f"{ba_dir}/train"

    # Região Centro — COS
    if area == "centro":
        lulcs = [
            f"{base}/lulc/rasters/lulc_2007.tif",
            f"{base}/lulc/rasters/lulc_2010.tif",
            f"{base}/lulc/rasters/lulc_2015.tif",
            f"{base}/lulc/rasters/lulc_2018.tif",
        ]

        burncos = [
            f"{ba_dir}/raster_count/rst_ba_2008_2009.tif",
            f"{ba_dir}/raster_count/rst_ba_2010_2014.tif",
            f"{ba_dir}/raster_count/rst_ba_2015_2017.tif",
            f"{ba_dir}/raster_count/rst_ba_2018_2024.tif",
        ]

        weights = [2, 5, 3, 7]

    # Extremadura — SIOSE e SIOSE AR
    elif area == "extremadura":
        lulcs = [
            f"{base}/lulc/rasters/lulc_siose_2005.tif",
            f"{base}/lulc/rasters/lulc_siose_2009.tif",
            f"{base}/lulc/rasters/lulc_siose_2011.tif",
            f"{base}/lulc/rasters/lulc_siose_2014.tif",
            f"{base}/lulc/rasters/lulc_siose_ar_2017.tif",
            f"{base}/lulc/rasters/lulc_siose_ar_2020.tif",
        ]

        burncos = [
            f"{ba_dir}/raster_count/rst_ba_2008.tif",
            f"{ba_dir}/raster_count/rst_ba_2009_2010.tif",
            f"{ba_dir}/raster_count/rst_ba_2011_2013.tif",
            f"{ba_dir}/raster_count/rst_ba_2014_2016.tif",
            f"{ba_dir}/raster_count/rst_ba_2017_2019.tif",
            f"{ba_dir}/raster_count/rst_ba_2020_2024.tif",
        ]

        weights = [1, 2, 3, 3, 3, 5]


dem = _vars["dem"]

os.makedirs(ws, exist_ok=True)

out_susc = f"{ws}/res_lri.tif"
out_prob = f"{ws}/wprobability.tif"
out_peri = f"{ws}/res_perigosity.tif"


print("Cenário:", scenario)
print("Área:", area)
print("Período:", train_start, "-", train_end)
print("LULC:", len(lulcs))

In [ ]:
from lri_sum import HeuristicLriCount
lri = HeuristicLriCount(ws, dem, loc=None)

In [ ]:
# Import topographic variables

lri.import_topo_vars(dem=_vars['dem'], slope=_vars['slope'])

# Import events
lri.import_events(bareas)

In [ ]:
# Import LULC variables
lri.import_lulc_vars(lulcs, weights, burncos)

In [ ]:
# Generate LRI
lri_res = lri.calc_lri(out=out_susc)

In [ ]:
lri.export_vars_lri(os.path.dirname(out_susc), 'lri')

In [ ]:
from glob import glob
import os
import grass.script as gs

from glass.rst.stats.grs import count_regionshp
from glass.rst.alg import grsrstcalc
from glass.it.rst import grs_to_rst


BASE_PROBABILITY = 0.01


def shp_year(path):
    return int(
        os.path.basename(path)
        .replace("aa_", "")
        .replace(".shp", "")
    )


train_shps = sorted(
    shp
    for shp in glob(f"{bareas_by_year}/aa_*.shp")
    if train_start <= shp_year(shp) <= train_end
)


n_years = train_end - train_start + 1


print(
    "Anos utilizados:",
    [shp_year(shp) for shp in train_shps]
)

print(
    "Número de anos:",
    n_years
)


# Garantir que a região computacional coincide com a suscetibilidade
gs.run_command(
    "g.region",
    raster=lri.final_lri,
    align=lri.final_lri
)


# Remover qualquer máscara GRASS deixada por operações anteriores
mask = gs.find_file("MASK", element="cell")

if mask.get("name"):
    gs.run_command("r.mask", flags="r")


# Probabilidade bruta: número de anos ardidos / número total de anos
raw_probability = count_regionshp(
    train_shps,
    "rst_wildfireprob_raw",
    return_prob=True,
    nnprob=n_years,
)


lri.wildfireprob = grsrstcalc(
    (
        f"if(isnull({lri.final_lri}), null(), "
        f"if(isnull({raw_probability}), {BASE_PROBABILITY}, "
        f"if({raw_probability} == 0, {BASE_PROBABILITY}, "
        f"{raw_probability})))"
    ),
    "rst_wildfireprob"
)


grs_to_rst(
    lri.wildfireprob,
    out_prob,
    dtype="Float64",
    nodata=-1,
)

In [ ]:
from glass.rst.alg import grsrstcalc
from glass.it.rst import grs_to_rst


lri.perigosity = grsrstcalc(
    (
        f"{lri.final_lri} "
        f"* {lri.wildfireprob}"
    ),
    "rst_perigosity"
)


grs_to_rst(
    lri.perigosity,
    out_peri,
    dtype="Float64",
    nodata=-1,
)